In [4]:
import os
# OMP/OPENBLAS 스레드 제한을 두지 않음: 후보 21,000여 개에 획득함수를 전수 평가하므로
# 1스레드로 묶으면 심각한 병목이 됨. (MKL_SERVICE_FORCE_INTEL만 호환성 위해 유지)
os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"

import time
import torch
import numpy as np
import pandas as pd

# ==========================================================
# FIXEDNOISEGP IMPORT (BoTorch 버전별 대응)
#   botorch 0.18+ 에서 FixedNoiseGP 클래스가 제거되고 그 기능이
#   SingleTaskGP(train_Yvar=...) 로 통합됨. train_Yvar를 주면 관측 노이즈가
#   고정(FixedNoiseGaussianLikelihood)되고, 안 주면 노이즈를 학습함.
# ==========================================================
try:
    from botorch.models import FixedNoiseGP          # botorch <= 0.17
except ImportError:
    from botorch.models import SingleTaskGP as FixedNoiseGP   # botorch >= 0.18

from botorch.models import ModelListGP
from gpytorch.mlls import SumMarginalLogLikelihood
from gpytorch.kernels import ScaleKernel, MaternKernel
from botorch.models.transforms import Normalize, Standardize
from botorch.acquisition.multi_objective.logei import qLogNoisyExpectedHypervolumeImprovement
from botorch.acquisition.multi_objective.objective import IdentityMCMultiOutputObjective
from botorch.utils.multi_objective.hypervolume import Hypervolume
from botorch.utils.multi_objective.pareto import is_non_dominated
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.fit import fit_gpytorch_mll

# ==========================================================
# CONFIGURATION
# ==========================================================
config = {
    "num_objectives": 3,
    "num_variables": 20,       # 4/6/8/12/16/20 중 선택
    "maximize": [True, False, False],   # ESW 최대 / Ehull 최소 / Erxn 최소
    "batch_q": 20,             # 세대당 선택 개수
    "n_generations": 10,       # 총 세대 수 (20 × 10 = 200개 추가 관측)
    "random_seed": 42,         # 42, 21, 10, 5
    "default_noise": 1e-4,     # 목적함수 분산 대비 "상대" 비율
    "pca_npz_path": "SevenNet/sevennet_pca_out/sevennet_pca_20d.npz",
    "gen0_xlsx":    "mobo_init/gen0_seeds.xlsx",
    "gen0_sheet":   "seed00",
    "out_xlsx":     "mobo_result/run_sevennet_seed00_dim=20.xlsx",
    # qLogNEHVI는 후보를 X_baseline(관측 전체)과 합쳐 posterior를 계산하므로
    # 커널 행렬이 (batch × n_obs²)로 커짐. OOM이 나면 아래 세 값을 줄일 것.
    "scan_batch_size": 4096,   # Stage 1 전수조사 배치
    "greedy_pool_size": 512,   # Stage 2로 넘길 상위 후보 수
    "stage2_chunk": 64,        # Stage 2 평가 청크 크기
    "ref_margin": 0.10,        # ref_point 여유폭 (풀 축범위의 10%)
}

torch.manual_seed(config["random_seed"])
np.random.seed(config["random_seed"])
dtype = torch.double
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}, dtype: {dtype}")
os.makedirs(os.path.dirname(config["out_xlsx"]), exist_ok=True)

# ==========================================================
# DATA LOADING — npz(전체 풀) + 엑셀(초기 200개 id)
# ==========================================================
d          = np.load(config["pca_npz_path"], allow_pickle=False)
ids_all    = d["ids"]
X_all_np   = d["embeddings"][:, :config["num_variables"]].astype(np.float64)
Y_all_np   = d["targets"].astype(np.float64)          # 원단위(raw), 부호 반전 전
obj_names  = [str(s) for s in d["target_names"]]
N_pool     = len(ids_all)
print(f"\n📦 pool: X={X_all_np.shape}, Y={Y_all_np.shape}")
print(f"   objectives = {obj_names}")
print(f"   maximize   = {config['maximize']}")

gen0    = pd.read_excel(config["gen0_xlsx"], sheet_name=config["gen0_sheet"])
id_col  = "mp_id" if "mp_id" in gen0.columns else gen0.columns[1]

def norm(s):   # npz는 'mp-1001069-GGA', 엑셀은 'mp-1001069' 형식 차이 흡수
    return pd.Series(s).astype(str).str.strip().str.replace(r"\.cif$|-GGA(\+U)?$", "", regex=True)

key_all = norm(ids_all)
assert key_all.duplicated().sum() == 0, "npz id 정규화 후 중복 존재"
pos = {k: i for i, k in enumerate(key_all)}

obs_idx = np.array([pos[k] for k in norm(gen0[id_col])], dtype=int)
assert len(obs_idx) == len(set(obs_idx.tolist())), "초기 집합에 중복 id"
print(f"📥 초기 관측: {len(obs_idx)}개 (sheet={config['gen0_sheet']})")

# ==========================================================
# 전역 텐서 (부호 반전 좌표계) — 루프에서 인덱싱으로만 사용
# ==========================================================
X_pool = torch.tensor(X_all_np, dtype=dtype, device=device)
Y_all  = torch.tensor(Y_all_np, dtype=dtype, device=device)
for i, mx in enumerate(config["maximize"]):
    if not mx:
        Y_all[:, i] = -Y_all[:, i]    # 부호 반전: 이후 전 좌표계 "클수록 좋음"

# ==========================================================
# 관측 노이즈 — gen0 관측으로 한 번만 산출하고 전 세대 고정
#   세대마다 갱신하면 노이즈 수준이 흔들려 세대 간 GP 비교가 어려워짐.
# ==========================================================
NOISE_VEC = torch.tensor(
    np.maximum(Y_all_np[obs_idx].var(axis=0, ddof=1) * config["default_noise"], 1e-12),
    dtype=dtype, device=device)
print(f"🔧 fixed noise (gen0 분산 × {config['default_noise']}): {NOISE_VEC.tolist()}")

# ==========================================================
# REFERENCE POINT — 후보 풀 전체 기준, 전 실험 고정
#   gen0 관측 기준으로 잡으면 seed마다 값이 달라 seed 간 HV 평균이 불가능함.
#   풀 전체의 축별 최악값을 쓰면 모델/seed/차원/세대에 무관하게 항상 동일하므로
#   seed 평균, 모델 간 비교, 달성률 정규화가 모두 가능해짐.
#   ※ ref_point는 GP 학습·획득함수 계산 어디에도 들어가지 않고 평가에만 쓰이므로
#     "정답을 훔쳐보는" 것이 아니라 "평가 자를 고정"하는 것임.
#   ※ min(0)은 축별 최솟값이라 실존 구조가 아닌 가상의 모서리점이며, 다목적에서는
#     이것이 정상. ref는 모든 파레토 점보다 각 축에서 엄격히 아래여야 부피가 생김.
# ==========================================================
m = config["ref_margin"]
ref_pool  = Y_all.min(0).values - m * (Y_all.max(0).values - Y_all.min(0).values)
ref_point = ref_pool            # 획득함수·주 지표에 사용

Y0 = Y_all[torch.tensor(obs_idx, device=device)]
ref_init = Y0.min(0).values - m * (Y0.max(0).values - Y0.min(0).values)   # 참고 기록용

print(f"📍 ref_pool (주 지표, 전 실험 고정): {[round(v,4) for v in ref_pool.tolist()]}")
print(f"📍 ref_init (참고 기록 전용)       : {[round(v,4) for v in ref_init.tolist()]}")

def hv_of(Yset, ref):
    with torch.no_grad():
        return float(Hypervolume(ref_point=ref).compute(Yset[is_non_dominated(Yset)]))

# 전수조사 HV — 풀 전체 파레토 프론트 기준 달성 가능 최대값.
#   현재HV / HV_MAX = 달성률. "몇 세대 만에 100%에 도달하는가"가
#   모델/차원 간 비교의 핵심 지표가 됨.
pareto_all = is_non_dominated(Y_all)
HV_MAX = hv_of(Y_all, ref_pool)
print(f"🎯 HV_max (전수조사, 풀 파레토 {int(pareto_all.sum())}개): {HV_MAX:.6f}")

# ref 이탈 관측 수 — ref보다 아래인 축이 하나라도 있으면 HV 기여 0.
#   ref가 풀 전체 기준이므로 원리상 항상 0이어야 함. 늘어나면 ref_margin 상향.
def count_outside_ref(Yset, ref):
    below = (Yset < ref)                        # (n, 3)
    n_any = int(below.any(dim=1).sum())         # 축 하나라도 이탈한 관측 수
    per_axis = below.sum(dim=0).tolist()        # 축별 이탈 수
    return n_any, per_axis

# ==========================================================
# MODEL BUILDER — 세대마다 재학습
#   배치 다출력 모델 대신 ModelListGP 사용:
#   botorch 0.18의 fit_gpytorch_mll은 다출력 모델일 때 covar_module을 직접 넘기면
#   커널 파라미터가 batch_shape=(3,)으로 확장되지 않아
#   "shape '[3, 1]' is invalid for input of size 1" 오류가 발생함.
#   ModelListGP는 목적함수마다 독립 GP를 두므로 배치 확장 자체가 불필요하고,
#   수학적으로 배치 다출력 모델과 동일함. qLogNEHVI가 그대로 지원함.
# ==========================================================
def build_and_fit(X_obs, Y_obs):
    Yvar = NOISE_VEC.expand_as(Y_obs).contiguous()
    submodels = []
    for j in range(config["num_objectives"]):
        submodels.append(
            FixedNoiseGP(
                train_X=X_obs,
                train_Y=Y_obs[:, j:j+1],
                train_Yvar=Yvar[:, j:j+1],
                covar_module=ScaleKernel(
                    MaternKernel(nu=2.5, ard_num_dims=config["num_variables"])
                ),
                input_transform=Normalize(config["num_variables"]),
                outcome_transform=Standardize(1),
            )
        )
    mdl = ModelListGP(*submodels).to(device=device, dtype=dtype)
    mll = SumMarginalLogLikelihood(mdl.likelihood, mdl)
    mdl.train()
    fit_gpytorch_mll(mll)
    mdl.eval()
    return mdl

# ==========================================================
# ACQUISITION — 후보 풀 전수조사
#
# 연속 공간 gradient 최적화 + 최근접 스냅을 쓰지 않는 이유:
#   (1) 스냅 오차: 연속 최적점은 실존 구조가 아님. 스냅된 "가장 가까운" 구조의
#       획득함수 값이 높다는 보장이 없음 — 획득함수가 뾰족한 영역일수록 이웃
#       구조의 획득값은 급락할 수 있고, 진짜 최적이었을 구조를 놓침.
#   (2) 중복 추천: 서로 다른 연속 최적점이 같은 구조로 스냅될 수 있음.
#   (3) 배치 붕괴: q joint로 최적화해도 스냅 과정에서 각 점이 독립적으로 이동하므로
#       배치의 joint 최적성이 깨짐.
#   (4) gradient 최적화 자체의 국소해 불안정성.
# 실험 가능한 후보는 어차피 "CIF 풀의 임베딩"이라는 유한 이산 집합이므로 연속 공간을
# 우회할 이유가 없음. 풀 전체에 획득함수를 직접 평가하면 위 문제가 원천 소멸함:
#   - 추천 벡터 = 풀의 실존 벡터 → 매칭 거리 0
#   - 전수조사이므로 풀 안에서는 전역 최적 보장
#   - 2단계 순차 탐욕(X_pending)으로 배치 다양성 확보
# ==========================================================
def select_batch(mdl, X_obs, active_idx):
    """active_idx: 아직 관측되지 않은 풀 인덱스(LongTensor).
       반환: 선택 인덱스, 조건부 획득값, stage1 최대값"""
    sampler = SobolQMCNormalSampler(sample_shape=torch.Size([64]))
    acq_func = qLogNoisyExpectedHypervolumeImprovement(
        model=mdl,
        ref_point=ref_point.tolist(),
        X_baseline=X_obs,
        objective=IdentityMCMultiOutputObjective(),
        prune_baseline=True,
        sampler=sampler,
    )
    X_act = X_pool[active_idx]

    # ------------------------------------------------------
    # [1단계] 전수조사
    #   배치별 top-k를 병합하지 않고 전체 값을 모은 뒤 한 번에 전역 top-k를 취함.
    #   배치별로 자르면 유망 후보가 한 배치에 몰렸을 때 뒷순위가 잘리고 나쁜 배치의
    #   후보가 대신 살아남아 "풀 안에서의 전역 최적"이 깨짐.
    #   획득값 2만여 개는 float64로 ~170KB이므로 전부 모아도 부담 없음.
    #   부수 효과: scan_batch_size가 결과에 영향을 주지 않음.
    # ------------------------------------------------------
    bs = config["scan_batch_size"]
    vals_list = []
    with torch.no_grad():
        for i, s in enumerate(range(0, X_act.shape[0], bs)):
            vals_list.append(acq_func(X_act[s:s+bs].unsqueeze(1)))
            if device.type == "cuda" and i % 20 == 19:
                torch.cuda.empty_cache()
    all_vals = torch.cat(vals_list)
    del vals_list
    k = min(config["greedy_pool_size"], all_vals.shape[0])
    top_vals, top_pos = torch.topk(all_vals, k)
    stage1_max = top_vals[0].item()
    del all_vals
    if device.type == "cuda":
        torch.cuda.empty_cache()

    # ------------------------------------------------------
    # [2단계] 순차 탐욕 배치 선택 (X_pending 갱신)
    #   이미 뽑힌 후보를 X_pending에 넣으면 다음 후보의 점수가 "그 후보가 선택된 것을
    #   전제로 한 조건부 개선량"이 됨. 하이퍼볼륨은 합집합 부피라 겹치는 기여를 두 번
    #   세지 않으므로, 먼저 뽑힌 후보 주변의 가치가 자동으로 깎이고 argmax가 아직 안
    #   덮인 영역으로 밀려남 → q개 배치의 다양성 확보.
    #   X_baseline이 관측 수만큼 커지므로 stage2_chunk 단위로 분할 평가 후 즉시 해제.
    # ------------------------------------------------------
    chunk = config["stage2_chunk"]
    chosen, chosen_acq = [], []
    remaining = top_pos.clone()
    with torch.no_grad():
        for _ in range(config["batch_q"]):
            if chosen:
                acq_func.set_X_pending(X_act[torch.stack(chosen)])
            vs = []
            for s in range(0, remaining.shape[0], chunk):
                vs.append(acq_func(X_act[remaining[s:s+chunk]].unsqueeze(1)))
            v = torch.cat(vs)
            del vs
            b = torch.argmax(v)
            chosen.append(remaining[b])
            chosen_acq.append(v[b].item())
            remaining = torch.cat([remaining[:b], remaining[b+1:]])
            del v
            if device.type == "cuda":
                torch.cuda.empty_cache()
        acq_func.set_X_pending(None)

    sel_local = torch.stack(chosen)
    return active_idx[sel_local].cpu().numpy(), chosen_acq, stage1_max

# ==========================================================
# 세대 루프
# ==========================================================
def save_progress(sel_rows, hv_rows):
    """세대마다 즉시 저장 — 중단되어도 그 시점까지 결과 보존.
       엑셀이 열려 있으면 PermissionError가 나는데(Windows 파일 잠금),
       계산 결과는 메모리에 있으므로 CSV로 우회 저장하고 루프는 계속 진행."""
    try:
        with pd.ExcelWriter(config["out_xlsx"], engine="openpyxl") as w:
            pd.concat(sel_rows, ignore_index=True).to_excel(w, sheet_name="selected", index=False)
            pd.DataFrame(hv_rows).to_excel(w, sheet_name="hypervolume", index=False)
    except PermissionError:
        alt = config["out_xlsx"].replace(".xlsx", "_backup.csv")
        pd.concat(sel_rows, ignore_index=True).to_csv(alt, index=False, encoding="utf-8-sig")
        pd.DataFrame(hv_rows).to_csv(alt.replace(".csv", "_hv.csv"), index=False, encoding="utf-8-sig")
        print(f"   ⚠ 엑셀 잠김 → CSV 백업: {alt}")

observed   = obs_idx.copy()                 # 누적 관측 인덱스
sel_rows, hv_rows = [], []
t_start = time.time()

# gen0 기록
Y_obs0 = Y_all[torch.tensor(observed, device=device)]
hv0    = hv_of(Y_obs0, ref_pool)
n_out0, ax_out0 = count_outside_ref(Y_obs0, ref_pool)
print(f"\n🌈 gen0  HV={hv0:.6f}  (달성률 {hv0/HV_MAX*100:6.2f}%)"
      f"  [pareto {int(is_non_dominated(Y_obs0).sum())}/{len(observed)}]"
      f"  ref이탈 {n_out0} (축별 {ax_out0})")
hv_rows.append({
    "generation": 0, "n_obs": len(observed), "hv": hv0, "hv_delta": 0.0,
    "hv_max": HV_MAX, "hv_frac": hv0 / HV_MAX,
    "n_pareto": int(is_non_dominated(Y_obs0).sum()),
    "n_outside_ref": n_out0, "outside_per_axis": str(ax_out0),
    "max_acq_stage1": np.nan, "hv_init_ref": hv_of(Y_obs0, ref_init),
    "elapsed_sec": 0.0,
})

for gen in range(1, config["n_generations"] + 1):
    t0 = time.time()
    X_obs = X_pool[torch.tensor(observed, device=device)]
    Y_obs = Y_all[torch.tensor(observed, device=device)]

    # --- GP 재학습 (매 세대) ---
    mdl = build_and_fit(X_obs, Y_obs)

    # --- 후보 풀: 관측된 것 인덱스로 직접 제외 ---
    #   좌표거리(cdist + tol) 방식은 (a) 부동소수점 기록 오차에 의존하고,
    #   (b) 서로 다른 두 구조의 임베딩이 우연히 tol 안에 들어오면 멀쩡한 후보까지
    #   제외되므로 사용하지 않음.
    mask = torch.ones(N_pool, dtype=torch.bool, device=device)
    mask[torch.tensor(observed, device=device)] = False
    active_idx = torch.arange(N_pool, device=device)[mask]

    # --- 배치 선택 ---
    new_idx, cond_acq, stage1_max = select_batch(mdl, X_obs, active_idx)
    assert len(set(new_idx.tolist()) & set(observed.tolist())) == 0, "중복 선택 발생"

    # --- ground truth 반영 (retrospective: 선택 후에만 조회) ---
    hv_before = hv_of(Y_obs, ref_pool)
    observed  = np.concatenate([observed, new_idx])
    Y_obs_new = Y_all[torch.tensor(observed, device=device)]
    hv_after  = hv_of(Y_obs_new, ref_pool)
    n_out, ax_out = count_outside_ref(Y_obs_new, ref_pool)
    n_par = int(is_non_dominated(Y_obs_new).sum())
    el    = time.time() - t0

    print(f"gen{gen:2d}  HV {hv_before:.6f} → {hv_after:.6f} (Δ{hv_after-hv_before:+.6f})"
          f"  달성률 {hv_after/HV_MAX*100:6.2f}%"
          f"  pareto {n_par}/{len(observed)}  acq_max {stage1_max:+.4f}"
          f"  ref이탈 {n_out}  [{el:.1f}s]")

    # --- 기록 ---
    df = pd.DataFrame({"generation": gen, "id": ids_all[new_idx], "pool_index": new_idx,
                       "cond_acq": cond_acq})
    for j, nm in enumerate(obj_names):
        df[nm] = Y_all_np[new_idx, j]                    # 원단위 저장
    for i in range(config["num_variables"]):
        df[f"x{i+1}"] = X_all_np[new_idx, i]
    sel_rows.append(df)
    hv_rows.append({
        "generation": gen, "n_obs": len(observed), "hv": hv_after,
        "hv_delta": hv_after - hv_before,
        "hv_max": HV_MAX, "hv_frac": hv_after / HV_MAX,
        "n_pareto": n_par,
        "n_outside_ref": n_out, "outside_per_axis": str(ax_out),
        "max_acq_stage1": stage1_max, "hv_init_ref": hv_of(Y_obs_new, ref_init),
        "elapsed_sec": el,
    })
    save_progress(sel_rows, hv_rows)

# ==========================================================
# OUTPUT
# ==========================================================
hv_df = pd.DataFrame(hv_rows)
print(f"\n{'='*78}")
print(f"완료: {config['n_generations']}세대, 관측 {len(obs_idx)} → {len(observed)}개"
      f"  ({time.time()-t_start:.1f}s)")
print(f"HV      {hv_df['hv'].iloc[0]:.6f} → {hv_df['hv'].iloc[-1]:.6f}"
      f"  (총 Δ{hv_df['hv'].iloc[-1]-hv_df['hv'].iloc[0]:+.6f})")
print(f"달성률  {hv_df['hv_frac'].iloc[0]*100:.2f}% → {hv_df['hv_frac'].iloc[-1]*100:.2f}%"
      f"   (HV_max = {HV_MAX:.6f})")
print(f"파레토  {hv_df['n_pareto'].iloc[0]} → {hv_df['n_pareto'].iloc[-1]}")
reach = hv_df[hv_df["hv_frac"] >= 0.9999]
print(f"100% 도달: gen{int(reach['generation'].iloc[0])}" if len(reach) else "100% 미도달")
if hv_df["n_outside_ref"].iloc[-1] > 0:
    print(f"⚠ ref 이탈 관측 {hv_df['n_outside_ref'].iloc[-1]}개 — ref_margin 상향 검토")
print(f"💾 saved -> {config['out_xlsx']}")

# 마지막 세대 선택 결과 미리보기
print(f"\n===== gen{config['n_generations']} 선택 =====")
print(sel_rows[-1][["id"] + obj_names + ["cond_acq"]].to_string(index=False))

🚀 Using device: cuda, dtype: torch.float64

📦 pool: X=(21544, 20), Y=(21544, 3)
   objectives = ['ESW_width', 'energy_above_hull_eV_per_atom', 'worst_Erxn(eV/atom)']
   maximize   = [True, False, False]
📥 초기 관측: 200개 (sheet=seed00)
🔧 fixed noise (gen0 분산 × 0.0001): [9.275206571204537e-05, 1.623668647011905e-05, 1.7282346784443733e-05]
📍 ref_pool (주 지표, 전 실험 고정): [-0.9362, -8.3683, -8.3465]
📍 ref_init (참고 기록 전용)       : [-0.6011, -3.484, -3.4774]
🎯 HV_max (전수조사, 풀 파레토 2개): 718.566833

🌈 gen0  HV=484.886187  (달성률  67.48%)  [pareto 4/200]  ref이탈 0 (축별 [0, 0, 0])
gen 1  HV 484.886187 → 510.424439 (Δ+25.538252)  달성률  71.03%  pareto 4/220  acq_max +3.0054  ref이탈 0  [20.4s]
gen 2  HV 510.424439 → 510.424439 (Δ+0.000000)  달성률  71.03%  pareto 4/240  acq_max +2.0787  ref이탈 0  [24.9s]
gen 3  HV 510.424439 → 718.447207 (Δ+208.022768)  달성률  99.98%  pareto 4/260  acq_max +1.9059  ref이탈 0  [26.2s]
gen 4  HV 718.447207 → 718.447207 (Δ+0.000000)  달성률  99.98%  pareto 4/280  acq_max +1.4167  ref이탈 0  [29